<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/08_learning_signal_and_ignoring_noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 8: Learning Signal and Ignoring Noise - Introduction to Regularization and Batching**

This notebook summarizes Chapter 8 of *Grokking Deep Learning*:

1. Three-Layer Network on MNIST
2. Well, That Was Easy
3. Memorization versus Generalization
4. Overfitting in Neural Networks
5. Where Overfitting Comes From
6. The Simplest Regularization: Early Stopping
7. Industry Standard Regularization: Dropout
8. Why Dropout Works: Ensembling Works
9. Dropout in Code
10. Dropout Evaluated on MNIST
11. Batch Gradient Descent
12. Summary

The chapter returns to the MNIST digits with the three-layer network that has a `relu` hidden layer, and it recalls the earlier experiment in which one weight was frozen while the network kept training. A network that reproduces its training data perfectly is not necessarily a network that works on new data, and this chapter makes that point concrete. It then introduces **regularization**, the family of techniques that help a network to generalize: early stopping (briefly) and **dropout** (in detail). It closes with **batch gradient descent**, a way of training faster and more smoothly by averaging the weight updates of many examples.

The chapter opens with a remark attributed to John von Neumann: with four parameters one can fit an elephant, and with a fifth one can even make it wiggle its trunk.

---
## **1. Three-Layer Network on MNIST**

**Let us return to MNIST and try to classify it with the new network.**

The previous chapters established that neural networks model **correlation**, and that a hidden layer (the middle layer of a three-layer network) can even create *intermediate* correlation that helps with the task. The question is how one can know that the network is creating *good* correlation.

### **1.1 A Reminder: The Frozen Weight**

In the experiment with stochastic gradient descent and multiple inputs, one weight was frozen and the network was asked to keep training. The frozen weight nevertheless found "the bottom of the bowl": the bowl itself moved until the frozen value became optimal. Moreover, if the weight were unfrozen for further training, it would not learn anything, because the error had already fallen to 0 and, as far as the network was concerned, there was nothing left to learn.

This raises an uncomfortable question. Suppose the input of the frozen weight had really been important for predicting a baseball victory in the real world. The network may have found a way to predict the games of the *training set* accurately (minimizing error is what networks do) while it simply left out a valuable input.

### **1.2 Overfitting**

This phenomenon is called **overfitting**, and it is extremely common in neural networks. The author calls it their archnemesis: the more expressive power a network has (more layers and weights), the more prone it is to overfit. Research is described as an everlasting battle, in which people keep finding tasks that need more powerful layers and then have to work hard to keep these networks from overfitting.

This chapter therefore studies the basics of **regularization**, the key to combating overfitting. The starting point is the most powerful network considered (the three-layer network with a `relu` hidden layer), applied to the most challenging task (MNIST digit classification).

### **1.3 Training the Network**

The listing below trains the network on the **first 1,000 images** of MNIST and prints the progress of the training. In outline:

- **Data.** The images are flattened to 784 pixel values and divided by 255. The labels are turned into one-hot vectors of length 10. The test images and test labels are prepared in the same way, and are used in the next section.
- **Activation.** `relu` returns `x` for positive `x` and 0 otherwise. `relu2deriv` returns 1 for positive input and 0 otherwise (the listing writes both with `>=`).
- **Settings.** `alpha = 0.005`, `iterations = 350`, `hidden_size = 40`, 784 pixels per image and 10 labels. Both weight matrices are initialized with random values between -0.1 and 0.1.
- **Training loop.** For every image the network predicts (a `relu` hidden layer followed by a linear output layer), accumulates the squared error, counts whether the largest output matches the label, computes `layer_2_delta` as the target minus the prediction, passes it back through `weights_1_2.T` and multiplies it by `relu2deriv(layer_1)` to obtain `layer_1_delta`, and then updates both weight matrices. After each iteration, the average error and the fraction of correctly classified images are written on a single line.

*Data note:* the code expects the MNIST dataset, which the book loads through `keras.datasets.mnist`. Several later listings of this chapter reuse variables defined here (`images`, `labels`, `test_images`, `test_labels`, `np`, `sys`, and others), so the cells must be run in order.

*Note on a repair:* the book prints the `images, labels = (...)` statement with a line break after `reshape(1000,28*28) \`, and the second line starts directly with `255, y_train[0:1000])`. The division operator is missing, so the statement would be a syntax error. This notebook uses `/ 255, y_train[0:1000])` on the second line, which is the same scaling that the book applies to `test_images` a few lines later.

In [1]:
import sys, numpy as np
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28) \
                  / 255, y_train[0:1000])
one_hot_labels = np.zeros((len(labels),10))

for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test),28*28) / 255
test_labels = np.zeros((len(y_test),10))
for i,l in enumerate(y_test):
    test_labels[i][l] = 1

np.random.seed(1)
relu = lambda x:(x>=0) * x
relu2deriv = lambda x: x>=0
alpha, iterations, hidden_size, pixels_per_image, num_labels = \
                                        (0.005, 350, 40, 784, 10)
weights_0_1 = 0.2*np.random.random((pixels_per_image,hidden_size)) - 0.1
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)

    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0,weights_0_1))
        layer_2 = np.dot(layer_1,weights_1_2)
        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == \
                                        np.argmax(labels[i:i+1]))
        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T)\
                                    * relu2deriv(layer_1)
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    sys.stdout.write("\r"+ \
                     " I:"+str(j)+ \
                     " Error:" + str(error/float(len(images)))[0:5] +\
                     " Correct:" + str(correct_cnt/float(len(images))))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
 I:349 Error:0.108 Correct:1.0

The printed result of the last iteration is `I:349 Error:0.108 Correct:1.0`. Because each output overwrites the previous one (the line starts with `\r`), only the final state remains visible. The network has learned to predict the training data **perfectly**. The author's question is whether this is a reason to celebrate.

---
## **2. Well, That Was Easy**

**The neural network perfectly learned to predict all 1,000 images.**

In some ways this is a real victory. The network took 1,000 images and learned to correlate each input image with its correct label. It did so by iterating over the images, predicting, and adjusting every weight a little so that the next prediction would be better. After enough passes it reached a state in which all the images were predicted correctly.

A less obvious question is how well the network will do on an image that it has **not seen before**. MNIST contains many more images than the 1,000 used for training, and the variables `test_images` and `test_labels` hold such images. The next block runs the network on them and measures how well it classifies them. The condition `j % 10 == 0 or j == iterations-1` refers to the iteration counter `j` of the training loop, so in the book's setting this evaluation belongs to the training loop and is carried out every 10 iterations and at the last one. When the block is run on its own after the loop has finished (as in the cell below), `j` still holds its last value, 349, so the evaluation runs once, on the final weights.

*Note:* the book shows the output of this block as `Error:0.653 Correct:0.7073`, while the code writes the labels `Test-Err:` and `Test-Acc:`. The numbers are the same as those at the end of the book's training log in the next section.

In [2]:
if(j % 10 == 0 or j == iterations-1):
    error, correct_cnt = (0.0, 0)

    for i in range(len(test_images)):

        layer_0 = test_images[i:i+1]
        layer_1 = relu(np.dot(layer_0,weights_0_1))
        layer_2 = np.dot(layer_1,weights_1_2)

        error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == \
                                        np.argmax(test_labels[i:i+1]))
    sys.stdout.write(" Test-Err:" + str(error/float(len(test_images)))[0:5] +\
               " Test-Acc:" + str(correct_cnt/float(len(test_images))))
    print()

 Test-Err:0.653 Test-Acc:0.7073


The network did badly: it reached a **test accuracy of only 70.7%** (test error 0.653), although it had reached 100% accuracy on the training data. This surprising gap is the central point of the chapter.

The **test accuracy** is the accuracy of the network on data that it was *not* trained on. It matters because it simulates how the network will behave in the real world, where it only sees images it has not seen before. It is "the score that matters".

---
## **3. Memorization versus Generalization**

**Memorizing 1,000 images is easier than generalizing to all images.**

Training adjusts each weight in each matrix so that the network can take *specific inputs* and produce a *specific prediction*. A better question than "why does it fail on new images?" might therefore be: "if the network learns to predict its 1,000 training images perfectly, why does it work on other images at all?"

A fully trained network is only **guaranteed** to work well on a new image if that image is nearly identical to one from the training data, because the network has learned to map only very specific input configurations to outputs. When it is given something unfamiliar, it predicts essentially at random.

If this were the whole story, neural networks would be rather pointless: a network that works only on the data it was trained on is of no use, since the correct classifications of those datapoints are already known. Networks are useful only if they work on data whose answer is *not* already known.

There is, however, a way to fight this problem, and a clue is visible when the training and testing accuracy are printed side by side as the network trains (every 10 iterations). The book prints the complete log; the table below keeps a selection of its rows (the book itself omits the iterations between 140 and 210):

| Iteration | Train-Err | Train-Acc | Test-Err | Test-Acc |
|---:|---:|---:|---:|---:|
| 0 | 0.722 | 0.537 | 0.601 | 0.6488 |
| 10 | 0.312 | 0.901 | 0.420 | **0.8114** |
| 20 | 0.260 | 0.93 | 0.414 | 0.8111 |
| 50 | 0.204 | 0.966 | 0.437 | 0.7982 |
| 100 | 0.166 | 0.984 | 0.482 | 0.7706 |
| 140 | 0.149 | 0.991 | 0.508 | 0.758 |
| 210 | 0.127 | 0.998 | 0.544 | 0.7446 |
| 300 | 0.113 | 0.999 | 0.614 | 0.7183 |
| 349 | 0.108 | 1.0 | 0.653 | 0.7073 |

The training error keeps falling and the training accuracy keeps rising up to 100%, but the **test accuracy peaks early (about 81%) and then declines** to about 71%.

---
## **4. Overfitting in Neural Networks**

**Neural networks can get worse if you train them too much!**

The test accuracy rose during roughly the first 20 iterations and then slowly decreased as the network trained more and more, while the *training* accuracy was still improving. This is common in neural networks, and the book explains it through an analogy.

### **4.1 The Fork Mold**

Imagine making a mold of an ordinary dinner fork, not to produce forks but to **recognize** them: an object that fits into the mold is declared a fork, and one that does not fit is declared not to be a fork. You start from wet clay and a big bucket containing three-pronged forks, spoons, and knives. You press the forks into the same place of the clay, over and over, hundreds of times, until the clay holds an outline of a fork. When the clay has dried, no spoon and no knife fits, but every fork does. The mold seems to work.

Now someone hands you a **four-pronged** fork. Your mold contains a specific outline of three thin prongs, so the four-pronged fork does not fit, although it is still a fork. The reason is that the clay was molded only on three-pronged forks: it has **overfit** to the kinds of forks it was "trained" on.

### **4.2 The Same Phenomenon in the Network**

The parallel with the network is close. The weights of a neural network can be seen as a **high-dimensional shape** that molds itself around the shape of the data during training, learning to tell one pattern from another. The testing images are *slightly different* from the patterns of the training set, so the network fails on many of them.

A more formal definition follows: **a neural network that overfits is one that has learned the noise in the dataset instead of making its decisions based only on the true signal.**

---
## **5. Where Overfitting Comes From**

**What causes neural networks to overfit?**

Consider the fresh clay again, but now press only a **single** fork into very thick clay. The imprint has much less detail than the heavily imprinted mold. It is only a *very general shape of a fork*, a fuzzy imprint that may well be compatible with both three- and four-pronged forks.

Seen this way, the heavily imprinted mold became *worse* on the testing set as more forks were pressed into it, because it learned more and more detailed information about its training data. It then rejects anything that is even slightly different from what it repeatedly saw.

### **5.1 What Counts as Noise**

The detailed information that is incompatible with the test data is, in the fork analogy, the number of prongs. In images it is generally called **noise**, although in reality the notion is a bit more subtle. Consider the two dog pictures below.

> Two pictures of a dog: a photograph of a dog on a cushion (left) and a plain black silhouette (right)
>
> <img src="pictures/chapter 8/picture 8.1.png" alt="Figure 8.1" width="600"/>

Everything that makes a picture unique beyond the essence of "dog" is noise:

- In the left picture, the **pillow and the background** are noise. The middle of the dog, with its furry texture and color, is not noise, because it can help a classifier to identify a dog.
- In the right picture, the **empty black middle** of the dog is a form of noise too. It is the *edges* that tell us that this is a dog, and the black interior says nothing.

### **5.2 Training on Signal Only**

How can a network be made to train only on the **signal** (the essence of "dog") and to ignore the noise (everything irrelevant to the classification)? One way is **early stopping**. A large part of the noise lies in the fine-grained detail of an image, whereas most of the signal for objects is found in the general shape and perhaps the color.

---
## **6. The Simplest Regularization: Early Stopping**

**Stop training the network when it starts getting worse.**

To make a network ignore fine detail and capture only the general information in the data (the general shape of a dog or of an MNIST digit), one simply does not let it train long enough to learn that detail. In the fork analogy, it takes many forks imprinted many times to obtain the perfect outline of a three-pronged fork, whereas the first few imprints capture only a shallow outline. Networks behave in the same way. **Early stopping** is therefore the cheapest form of regularization, and it can be quite effective when one is in a pinch.

This leads to the subject of the chapter:

> **Regularization** is a subset of methods used to encourage generalization in learned models, often by increasing the difficulty for a model to learn the fine-grained details of the training data.

Regularization is thus a subfield of methods for getting a model to **generalize** to new datapoints instead of merely memorizing the training data. It helps the network learn the signal and ignore the noise, and it is a toolset available for building networks with these properties.

### **6.1 How Do We Know When to Stop?**

The only real way to know is to evaluate the model on data that is not in the training set. This is typically done with a further dataset called a **validation set**. If the *test* set were used to decide when to stop, one could end up **overfitting to the test set**. As a general rule, the test set is therefore not used to control training; a validation set is used instead.

---
## **7. Industry Standard Regularization: Dropout**

**The method: randomly turn off neurons (set them to 0) during training.**

**Dropout** is as simple as it sounds. During training, neurons of the network are randomly set to 0, and usually the deltas of the same nodes are zeroed during backpropagation as well (although technically this is not required). As a result, the network is trained exclusively on **random subsections** of itself.

The book states that dropout is generally accepted as the go-to, state-of-the-art regularization technique for the vast majority of networks. The method is simple and inexpensive, although the intuition for *why* it works is somewhat more involved.

> **Why dropout works (perhaps oversimplified):** dropout makes a big network act like a little one, by randomly training little subsections of the network at a time, and little networks do not overfit.

### **7.1 Capacity: Small Networks Cannot Overfit as Much**

The smaller a network is, the less it is able to overfit. A small network has little expressive power: it cannot latch on to the granular details (the noise) that are the usual source of overfitting, and it has room to capture only the large, obvious, high-level features.

The notion of **room** or **capacity** can be pictured with the clay analogy:

- Imagine the clay is made of sticky rocks the size of dimes. Such clay cannot take a good imprint of a fork. The rocks play the role of the weights: they form around the data and capture the patterns of interest. With only a few large stones, nuanced detail cannot be captured, because each stone is pushed on by a large part of the fork and thus more or less *averages* the shape, ignoring fine creases and corners.
- Now imagine clay made of very fine sand, that is, millions and millions of small stones that fit into every nook and cranny of the fork. This is what gives big networks the expressive power that they often use to overfit.

### **7.2 Getting the Best of Both**

How can one have the power of a large network and the resistance to overfitting of a small one? By taking the large network and **turning off nodes randomly**. When only a small part of a big network is in use, it behaves like a small network. Yet when this is done randomly across potentially millions of different subnetworks, the sum total of the whole network still keeps its expressive power.

---
## **8. Why Dropout Works: Ensembling Works**

**Dropout is a form of training a bunch of networks and averaging them.**

Neural networks always start out randomly, and since they learn by trial and error, every network learns a little differently. Two networks may learn equally effectively, but they are never exactly the same unless they happen to start from exactly the same state, by chance or by design.

This has an interesting consequence for overfitting: **no two networks overfit in exactly the same way**. Overfitting goes on only until every training image is predicted perfectly; at that moment the error is 0 and the network stops learning, even if the iterations continue. Because each network starts by predicting randomly and then adjusts its weights, each one inevitably makes different mistakes and receives different updates. This leads to a core concept:

> Although it is likely that large, unregularized neural networks will overfit to noise, it is unlikely that they will overfit to the *same* noise.

They do not overfit to the same noise because they start randomly and stop training as soon as they have learned enough noise to tell all the training images apart. For MNIST, a handful of random pixels that happen to correlate with the output labels is enough to overfit. This is contrasted by a perhaps even more important concept:

> Neural networks, even though they are randomly generated, still start by learning the biggest, most broadly sweeping features before learning much about the noise.

The takeaway is the following. If 100 randomly initialized networks are trained, each will tend to latch on to *different noise* but to a *similar broad signal*. When they make mistakes, they will often make **different** mistakes. If the networks were allowed to vote equally, their noise would tend to cancel out, and only what they all learned in common would remain: **the signal**. Dropout can be viewed as exactly this kind of ensemble: training a multitude of networks (the random subnetworks) and averaging them.

---
## **9. Dropout in Code**

**Here is how to use dropout in the real world.**

Dropout will be added to the hidden layer of the MNIST model so that **50% of its nodes are turned off at random** during training. Surprisingly, this is only a **three-line change** of the code. The book first shows a familiar snippet of the previous network logic with the dropout lines added (they are printed in bold in the book):

- `dropout_mask = np.random.randint(2,size=layer_1.shape)` creates a matrix of random 0s and 1s with the shape of `layer_1`;
- `layer_1 *= dropout_mask * 2` applies the mask to the hidden layer and rescales it;
- `layer_1_delta *= dropout_mask` applies the same mask to the delta of the hidden layer, so that the nodes that were switched off receive no update.

To apply dropout to a layer (here `layer_1`), its values are multiplied by a random matrix of 1s and 0s, which sets the corresponding nodes to 0. The `dropout_mask` follows what is called a **50% Bernoulli distribution**: each of its values is 1 half of the time and 0 for the remaining (1 - 50% = 50%) of the time.

### **9.1 Why Multiply by 2?**

The multiplication of `layer_1` by 2 may look peculiar. `layer_2` computes a weighted sum of the values of `layer_1`, and if half of those nodes are turned off, the sum is cut in half. `layer_2` would then become more sensitive to `layer_1`, like a person who leans closer to a radio whose volume is too low. At **test time**, however, dropout is no longer used, so the volume returns to normal, and this upsets the ability of `layer_2` to "listen" to `layer_1`. The remedy is to multiply `layer_1` by 1 divided by the fraction of nodes that are switched on, which here is 1 / 0.5 = 2. In this way the "volume" of `layer_1` is the same in training and in testing despite dropout.

*Notes on the snippet below:*

- It is a fragment, not a complete program. It reuses variables from the previous cells (`images`, `labels`, `layer_1`, `weights_1_2`, `alpha`, `error`, `correct_cnt`, `relu2deriv`, and others) and therefore runs only after the earlier cells have been executed. Running it performs a single training step for the image with index 0.
- The book prints the label argument of `correct_cnt` as `labels[i+i+1]`, whereas the complete listing below uses `labels[i:i+1]`. The snippet is kept exactly as printed; since `i = 0` it still runs (it reads the label of the second image).

In [3]:
i = 0
layer_0 = images[i:i+1]
dropout_mask = np.random.randint(2,size=layer_1.shape)

layer_1 *= dropout_mask * 2
layer_2 = np.dot(layer_1, weights_1_2)

error += np.sum((labels[i:i+1] - layer_2) ** 2)

correct_cnt += int(np.argmax(layer_2) == \
                   np.argmax(labels[i+i+1]))

layer_2_delta = (labels[i:i+1] - layer_2)
layer_1_delta = layer_2_delta.dot(weights_1_2.T)\
                * relu2deriv(layer_1)

layer_1_delta *= dropout_mask

weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

### **9.2 The Complete Dropout Listing**

The next listing puts the three dropout lines into the full training loop. It differs from the first network in a few settings: `iterations = 300` and `hidden_size = 100` (before, 350 and 40), while `alpha` stays at 0.005. It re-defines `relu` and `relu2deriv` as functions. Every 10 iterations (`j%10 == 0`) it evaluates the network on all the test images **without dropout** and prints the test error, the test accuracy, and the training error and accuracy of the current iteration (the latter are measured *with* dropout active).

*Notes:*

- The book prints the first line as `import numpy, sys` but then uses the alias `np`. The alias was defined in the first listing of this chapter (`import sys, numpy as np`), so the code runs when the cells are executed in order, and it is kept as printed.
- The data variables (`images`, `labels`, `test_images`, `test_labels`) also come from the first listing.

In [4]:
import numpy, sys
np.random.seed(1)
def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0

alpha, iterations, hidden_size = (0.005, 300, 100)
pixels_per_image, num_labels = (784, 10)

weights_0_1 = 0.2*np.random.random((pixels_per_image,hidden_size)) - 0.1
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0,0)
    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0,weights_0_1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = np.dot(layer_1,weights_1_2)

        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == \
                                        np.argmax(labels[i:i+1]))
        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if(j%10 == 0):
        test_error = 0.0
        test_correct_cnt = 0

        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = relu(np.dot(layer_0,weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)

            test_error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
            test_correct_cnt += int(np.argmax(layer_2) == \
                                    np.argmax(test_labels[i:i+1]))

        sys.stdout.write("\n" + \
                         "I:" + str(j) + \
                         " Test-Err:" + str(test_error/ float(len(test_images)))[0:5] +\
                         " Test-Acc:" + str(test_correct_cnt/ float(len(test_images)))+\
                         " Train-Err:" + str(error/ float(len(images)))[0:5] +\
                         " Train-Acc:" + str(correct_cnt/ float(len(images))))


I:0 Test-Err:0.641 Test-Acc:0.6333 Train-Err:0.891 Train-Acc:0.413
I:10 Test-Err:0.458 Test-Acc:0.787 Train-Err:0.472 Train-Acc:0.764
I:20 Test-Err:0.415 Test-Acc:0.8133 Train-Err:0.430 Train-Acc:0.809
I:30 Test-Err:0.421 Test-Acc:0.8114 Train-Err:0.415 Train-Acc:0.811
I:40 Test-Err:0.419 Test-Acc:0.8112 Train-Err:0.413 Train-Acc:0.827
I:50 Test-Err:0.409 Test-Acc:0.8133 Train-Err:0.392 Train-Acc:0.836
I:60 Test-Err:0.412 Test-Acc:0.8236 Train-Err:0.402 Train-Acc:0.836
I:70 Test-Err:0.412 Test-Acc:0.8033 Train-Err:0.383 Train-Acc:0.857
I:80 Test-Err:0.410 Test-Acc:0.8054 Train-Err:0.386 Train-Acc:0.854
I:90 Test-Err:0.411 Test-Acc:0.8144 Train-Err:0.376 Train-Acc:0.868
I:100 Test-Err:0.411 Test-Acc:0.7903 Train-Err:0.369 Train-Acc:0.864
I:110 Test-Err:0.411 Test-Acc:0.8003 Train-Err:0.371 Train-Acc:0.868
I:120 Test-Err:0.402 Test-Acc:0.8046 Train-Err:0.353 Train-Acc:0.857
I:130 Test-Err:0.408 Test-Acc:0.8091 Train-Err:0.352 Train-Acc:0.867
I:140 Test-Err:0.405 Test-Acc:0.8083 Train-Er

---
## **10. Dropout Evaluated on MNIST**

Recall that the network without dropout reached a test accuracy of **81.14%** and then fell back, finishing training at **70.73%**. With dropout, the network behaves differently. The book prints the output below every 10 iterations; the table keeps a selection of its rows:

| Iteration | Test-Err | Test-Acc | Train-Err | Train-Acc |
|---:|---:|---:|---:|---:|
| 0 | 0.641 | 0.6333 | 0.891 | 0.413 |
| 10 | 0.458 | 0.787 | 0.472 | 0.764 |
| 20 | 0.415 | 0.8133 | 0.430 | 0.809 |
| 60 | 0.412 | **0.8236** | 0.402 | 0.836 |
| 100 | 0.411 | 0.7903 | 0.369 | 0.864 |
| 200 | 0.405 | 0.8036 | 0.347 | 0.893 |
| 250 | 0.395 | 0.8182 | 0.320 | 0.899 |
| 290 | 0.399 | 0.8181 | 0.301 | 0.908 |

Dropout leads to two observations:

- The network **peaks at a test accuracy of 82.36%** and it does not overfit nearly as badly: it finishes training (at the last printed iteration, 290) with a test accuracy of **81.81%**, compared with 70.73% without dropout.
- Dropout also **slows down the training accuracy**, which had previously gone straight to 100% and stayed there. Here it is only about 0.91 at iteration 290.

This shows what dropout really is: **noise**. It makes it harder for the network to fit the training data, like running a marathon with weights on one's legs. Training is harder, but when the weights come off for the big race, one runs noticeably faster because the training was for something much more difficult.

---
## **11. Batch Gradient Descent**

**A method for increasing the speed of training and the rate of convergence.**

In the context of this chapter, the book briefly applies a concept introduced several chapters earlier: **mini-batched stochastic gradient descent**. It is not treated in detail because it is largely taken for granted in neural network training, and it is a simple idea that does not become more advanced even in state-of-the-art networks.

Until now the network was trained **one example at a time**, with a weight update after each example. Now it is trained on **100 examples at a time**, and the weight update is the **average** over those 100 examples. The output of such a run (every 10 iterations; a selection of rows) is:

| Iteration | Test-Err | Test-Acc | Train-Err | Train-Acc |
|---:|---:|---:|---:|---:|
| 0 | 0.815 | 0.3832 | 1.284 | 0.165 |
| 10 | 0.568 | 0.7173 | 0.591 | 0.672 |
| 30 | 0.485 | 0.7793 | 0.498 | 0.754 |
| 60 | 0.452 | 0.7995 | 0.452 | 0.799 |
| 100 | 0.448 | 0.793 | 0.447 | 0.796 |
| 200 | 0.435 | 0.7972 | 0.416 | 0.829 |
| 240 | 0.430 | 0.8047 | 0.397 | 0.844 |
| 290 | 0.428 | 0.7975 | 0.383 | 0.851 |

The **training accuracy follows a smoother trend** than before. Averaging the weight updates consistently produces this effect, because the weight updates generated by individual training examples are very noisy, so averaging them gives a smoother learning process.

### **11.1 The Listing**

The listing below combines batching with the dropout of the previous section:

- `batch_size = 100`, and the loop over `i` now runs `len(images) / batch_size` times, so each pass takes a slice of 100 images, `images[batch_start:batch_end]`, as `layer_0`.
- The forward pass, including the dropout mask, works on the whole batch at once. The `for k` loop counts how many of the 100 predictions are correct (see also the note on indentation below).
- `layer_2_delta` is divided by `batch_size`, so the weight updates are *averages* over the batch.
- The test evaluation still goes through the test images one at a time.

*Notes on the printed listing:*

- **The listing ends inside the test loop.** The book's code stops after the line `layer_2 = np.dot(layer_1, weights_1_2)` of the test loop. The accumulation of the test error and correct count and the line that writes the results are not reprinted (they correspond to the ones in the dropout listing of the previous section). The cell below is kept exactly as printed, so it runs but prints nothing; the training log shown above is the book's output.
- **The value of `alpha` and the layout of the update.** The listing sets `alpha = 0.001`, whereas the text says that `alpha` is "20 times larger than before" (the previous value was 0.005). In addition, on the printed page the lines from `layer_2_delta = ...` down to the two weight updates are indented at the same level as the body of the `for k` loop, that is, inside it. The update is then carried out `batch_size` (100) times for every batch with practically the same averaged delta, whereas the text describes a single averaged update per batch. Repeating a step of size 0.001 one hundred times is roughly one step with `alpha` = 0.1, which is 20 times 0.005, and this may be what the remark in the text refers to. The notebook keeps both the printed value and the printed layout unchanged, because the code runs as printed. As a consequence, running this cell takes noticeably longer than the speed-up described below would suggest.
- The listing uses the data variables of the first listing (`images`, `labels`, `test_images`, `test_labels`).

In [5]:
import numpy as np
np.random.seed(1)

def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0

batch_size = 100
alpha, iterations = (0.001, 300)
pixels_per_image, num_labels, hidden_size = (784, 10, 100)

weights_0_1 = 0.2*np.random.random((pixels_per_image,hidden_size)) - 0.1
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size),((i+1)*batch_size))

        layer_0 = images[batch_start:batch_end]
        layer_1 = relu(np.dot(layer_0,weights_0_1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = np.dot(layer_1,weights_1_2)

        error += np.sum((labels[batch_start:batch_end] - layer_2) ** 2)
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == \
                               np.argmax(labels[batch_start+k:batch_start+k+1]))

            layer_2_delta = (labels[batch_start:batch_end]-layer_2) \
                            /batch_size
            layer_1_delta = layer_2_delta.dot(weights_1_2.T)* \
                            relu2deriv(layer_1)
            layer_1_delta *= dropout_mask

            weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
            weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if(j%10 == 0):
        test_error = 0.0
        test_correct_cnt = 0

        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = relu(np.dot(layer_0,weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)

### **11.2 Why Batching Is Faster, and Why a Larger `alpha` Is Possible**

The first thing noticed when running this code is that it is **much faster**. Each `np.dot` call now performs 100 vector dot products at once, and CPU architectures are much faster at dot products batched in this way.

There is more going on, however. The text says that `alpha` can be increased for an interesting reason. Imagine trying to find a city with a very wobbly compass. If you look down once, take a single heading, and run 2 miles, you will most likely end up far off course. If instead you take 100 headings and average them, running 2 miles will probably take you in roughly the right direction. In the same way, because the update is an **average of a noisy signal** (the average weight change over 100 training examples), the network can take **bigger steps**.

In practice, batch sizes generally range **from 8 to as high as 256**. Researchers usually pick numbers more or less at random until they find a `batch_size`/`alpha` pair that works well.

---
## **12. Summary**

- **Overfitting** is the archnemesis of neural networks: a network can predict its training data perfectly (here 100% on 1,000 MNIST images) and still do poorly on new data (70.7% test accuracy). The more expressive power a network has, the more prone it is to overfit.
- The **test accuracy**, measured on data the network was not trained on, is the score that matters, because it simulates real-world use. In the experiment it peaked at about 81% early in training and then declined while the training accuracy kept rising.
- A network that overfits has learned the **noise** of the training set (details such as the number of prongs on a fork, or the pillow and background of a dog photograph) instead of the **signal** (the general shape that defines the class). As in the fork-mold analogy, imprinting the same examples again and again makes the "mold" more detailed and less tolerant of new variations.
- **Regularization** is the family of methods that encourage generalization, often by making it harder for the model to learn the fine details of the training data.
- **Early stopping** is the cheapest regularizer: stop training when the network starts getting worse. A separate **validation set**, not the test set, should be used to decide when to stop.
- **Dropout** randomly sets nodes to 0 during training (and applies the same mask to their deltas). It makes a big network behave like many little ones, which cannot overfit as much, and it works like an **ensemble**: networks overfit to different noise but learn a similar broad signal, so the noise cancels out. Because dropout halves the sum seen by the next layer, the kept values are multiplied by 1 / (fraction kept), here 2, so that training and testing see the same "volume".
- In the experiment, dropout reached 82.36% test accuracy at its best and finished at 81.81% instead of 70.73%, while the training accuracy rose more slowly. Dropout is essentially noise that makes training harder, like a marathon run with weights on the legs.
- **Batch gradient descent** trains on groups of examples (here 100) and averages their weight updates. It runs much faster because `np.dot` handles the whole batch at once, it gives a smoother training curve because single examples produce noisy updates, and it allows a larger `alpha`. Typical batch sizes lie between 8 and 256, and a good `batch_size`/`alpha` pair is usually found by trying.
- Dropout and batching are two of the most widely used methods for improving the accuracy and the training speed of almost any neural architecture. The following chapters turn from such universally applicable tools to special-purpose architectures for modeling specific kinds of phenomena in data.